# Image stats and augmentation

Two questions before training a vision model: *what shape is this data actually
in*, and *what does my augmentation do to it*. Both are cheap to answer and both
are usually skipped, which is why the first training run so often produces a
number nobody can explain.

In [ ]:
import orinth
import polars as pl

DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type in {"classification", "object_detection", "segmentation"}
)
frame = orinth.datasets.load(DATASET_ID, "train", limit=1000)
print(orinth.datasets.get(DATASET_ID).name, "|", frame.height, "items")

frame.select(
    pl.col("width"), pl.col("height"), (pl.col("width") / pl.col("height")).alias("aspect")
).describe()

## Resolution spread

A single resize is fine when the spread is narrow. When it is not, the small end
is being upscaled into blur and the wide end is being squashed — and `letterbox`
or per-aspect bucketing is the fix, not a bigger model.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].scatter(frame["width"], frame["height"], s=6, alpha=0.4)
axes[0].set_xlabel("width")
axes[0].set_ylabel("height")
axes[1].hist((frame["width"] / frame["height"]).to_list(), bins=30)
axes[1].set_xlabel("aspect ratio")
figure.tight_layout()
plt.show()

## Brightness, per class

Systematic brightness differences between classes are a leak: the model learns
the capture condition instead of the subject, scores well on your split, and
collapses on anyone else's data.

In [ ]:
import numpy as np
from PIL import Image

brightness = []
for row in frame.head(200).iter_rows(named=True):
    with Image.open(row["path"]) as handle:
        pixels = np.asarray(handle.convert("L").resize((64, 64)), dtype="float32")
    brightness.append({"label": row["label"] or "unlabelled", "mean": float(pixels.mean())})

pl.DataFrame(brightness).group_by("label").agg(
    pl.len().alias("items"), pl.col("mean").mean().round(1).alias("brightness")
).sort("brightness")

## What the augmentation actually does

Preview before you train. An augmentation strong enough to destroy the label —
a vertical flip on a dataset where orientation *is* the class — is invisible in
the config and obvious in a grid.

In [ ]:
import tensorflow as tf

augment = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.06),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.2),
    ]
)

with Image.open(frame["path"][0]) as handle:
    original = np.asarray(handle.convert("RGB").resize((192, 192)), dtype="float32") / 255.0

figure, axes = plt.subplots(1, 5, figsize=(12, 2.6))
axes[0].imshow(original)
axes[0].set_title("original", fontsize=9)
for axis in axes[1:]:
    axis.imshow(np.clip(augment(original[None, ...], training=True)[0].numpy(), 0, 1))
    axis.set_title("augmented", fontsize=9)
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## Boxes, if this is a detection set

`annotations` rides on every image row. Very small boxes are the usual reason a
detector misses things: below roughly 32px on the model's input grid there is
not much left to detect after the stride.

In [ ]:
boxes = []
for row in frame.iter_rows(named=True):
    for annotation in row["annotations"] or []:
        box = annotation.get("bbox")
        if box:
            boxes.append({"label": annotation["label"], "w": box["width"], "h": box["height"]})
        elif annotation.get("polygon"):
            # A segmentation dataset carries no box; the polygon's extent is the
            # same measurement and is what a detector would have to find.
            points = annotation["polygon"]
            xs = [x for x, _ in points]
            ys = [y for _, y in points]
            boxes.append({"label": annotation["label"], "w": max(xs) - min(xs), "h": max(ys) - min(ys)})

if boxes:
    print(
        pl.DataFrame(boxes)
        .with_columns((pl.col("w") * pl.col("h")).sqrt().alias("size"))
        .group_by("label")
        .agg(pl.len().alias("boxes"), pl.col("size").median().round(1).alias("median_px"))
        .sort("boxes", descending=True)
    )
else:
    print("No annotations in this split — this is a classification dataset.")